Checking for GPU availability

In [2]:
!nvidia-smi
!nvcc --version

Sun Feb  8 14:05:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Step 2: Install llama.cpp Python Bindings

In [3]:
# Install llama-cpp-python with GPU support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir --force-reinstall --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 143.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 238.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 378.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 257.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 215.5 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=52139489 sha256=6cb2112010329d62cb028f5493bd57a6988f2924332e03a61c1024e00d7377f9
  Stored in directory: /tmp/pip-ephem-wheel-cache-7fvqol4o/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installatio

In [4]:
# Download model from Hugging Face
!pip install huggingface_hub
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    local_dir="./models"
)

print(f"Model downloaded to: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Llama-3.2-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

Model downloaded to: models/Llama-3.2-3B-Instruct-Q4_K_M.gguf


In [5]:
from llama_cpp import Llama

# Load model with GPU acceleration
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # Offload all layers to GPU
    n_ctx=2048,           # Context window (tokens)
    verbose=True          # Show loading info
)

print("✓ Model loaded successfully")

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14992 MiB free
llama_model_loader: loaded meta data with 35 key-value pairs and 255 tensors from models/Llama-3.2-3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str      

✓ Model loaded successfully


Using gguf chat template: {{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date

In [6]:
import time

# Prepare prompt
prompt = "Explain what a GPU is in one sentence."

# Generate response
start_time = time.time()

response = llm(
    prompt,
    max_tokens=100,       # Max length of response
    temperature=0.7,      # Creativity (0=deterministic, 1=creative)
    stop=["User:", "\n\n"]  # Stop tokens
)

end_time = time.time()

# Extract results
output_text = response['choices'][0]['text']
tokens_generated = response['usage']['completion_tokens']
time_taken = end_time - start_time
tokens_per_sec = tokens_generated / time_taken

# Display results
print(f"Response: {output_text}")
print(f"\n--- Performance ---")
print(f"Tokens generated: {tokens_generated}")
print(f"Time taken: {time_taken:.2f}s")
print(f"Speed: {tokens_per_sec:.1f} tokens/sec")

llama_perf_context_print:        load time =     220.43 ms
llama_perf_context_print: prompt eval time =     220.24 ms /    11 tokens (   20.02 ms per token,    49.94 tokens per second)
llama_perf_context_print:        eval time =     536.50 ms /    37 runs   (   14.50 ms per token,    68.97 tokens per second)
llama_perf_context_print:       total time =     800.82 ms /    48 tokens
llama_perf_context_print:    graphs reused =         35


Response:  The GPU (Graphics Processing Unit) is a specialized electronic circuit designed to rapidly manipulate and alter memory to accelerate the creation of images on a display device, such as a computer monitor or television.

--- Performance ---
Tokens generated: 38
Time taken: 0.80s
Speed: 47.2 tokens/sec


In [7]:
# Check VRAM usage
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
2526 MiB, 15360 MiB


In [8]:
def chat(user_message):
    response = llm(
        f"User: {user_message}\nAssistant:",
        max_tokens=200,
        temperature=0.7,
        stop=["User:"]
    )
    return response['choices'][0]['text'].strip()

# Try it
print(chat("What's the capital of France?"))
print(chat("Write a haiku about coding."))

Llama.generate: 1 prefix-match hit, remaining 11 prompt tokens to eval
llama_perf_context_print:        load time =     220.43 ms
llama_perf_context_print: prompt eval time =      18.50 ms /    11 tokens (    1.68 ms per token,   594.50 tokens per second)
llama_perf_context_print:        eval time =      96.51 ms /     8 runs   (   12.06 ms per token,    82.90 tokens per second)
llama_perf_context_print:       total time =     125.37 ms /    19 tokens
llama_perf_context_print:    graphs reused =          7


The capital of France is Paris.


Llama.generate: 3 prefix-match hit, remaining 9 prompt tokens to eval
llama_perf_context_print:        load time =     220.43 ms
llama_perf_context_print: prompt eval time =      22.19 ms /     9 tokens (    2.47 ms per token,   405.62 tokens per second)
llama_perf_context_print:        eval time =     288.93 ms /    23 runs   (   12.56 ms per token,    79.60 tokens per second)
llama_perf_context_print:       total time =     337.08 ms /    32 tokens
llama_perf_context_print:    graphs reused =         21


Here is a haiku about coding:

Lines of code unfold
 Logic in my tired eyes
Code's gentle delight


End